In [1]:
from os import listdir
from experiment_main import *

def get_crossover_results(
    df_test,
    df_valid,
    name_mode,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
):
    dff_test, current_bet_list_optim = naked_crossover(
        df_test,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    check_index_first = find_best_index(
        dff_test, 1, "net", name_mode, alpha_range, gamma_range
    )
    check_index_second = find_best_index(
        dff_test, 2, "net", name_mode, alpha_range, gamma_range
    )
    check_index_third, best_list = find_best_index(
        dff_test, 3, "net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth = find_best_index(
        dff_test, 4, "net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm
    base_histogram = dff_test[dff_test["his"] == f"{name_mode} a: 0"].index[0]

    # print(check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram)

    dff_valid, current_bet_list_optim = naked_crossover(
        df_valid,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    return (
        dff_valid,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    )

def add_df_to_list_detailed(
    output_list,
    value_df,
    index_first,
    index_second,
    index_third,
    index_fourth,
    base_his,
):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_second_list = list(value_df.iloc[index_second])
    index_third_list = list(value_df.iloc[index_third])
    index_fourth_list = list(value_df.iloc[index_fourth])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_second_list[i]
        output_list[2][i] += index_third_list[i]
        output_list[3][i] += index_fourth_list[i]
        output_list[4][i] += index_base_list[i]


def add_df_to_list_overall(output_list, value_df, index_first, base_his):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_base_list[i]


def get_data_rule(rule_data, len_return_list):
    ticker_length = 1
    return_list = [[] for i in range(len_return_list)]

    [
        "net",  # 0
        "netbet",  # 1
        "netpercent",  # 2
        "bets",  # 3
        "wins",  # 4
        "longs",  # 5
        "long_wins",  # 6
        "long_profit",  # 7
        "shorts",  # 8
        "short_wins",  # 9
        "short_profit",  # 10
        "long_only_net",  # 11
        "long_only_bets",  # 12
        "long_only_wins",  # 13
        "long_only_net_other",  # 14
        "long_only_bets_other",  # 15
        "long_only_wins_other",  # 16
        "his",  # 17
    ]
    # [0'net' , 1'netbet' , 2'netpercent' , 3'bets' , 4'wins' , 5'longs' , 6'long_wins' , 7'long_profit' , 8'shorts' , 9'short_wins' , 10'short_profit',
    # 11"long_only_net", 12"long_only_bets", 13"long_only_wins", 14"long_only_net_other", 15"long_only_bets_other", 16"long_only_wins_other", 17'his']

    # avg return for ticker , bets , win/loss , avg long return for ticker , longs , long win ratio , avg short return , short win ratio,
    # 0 / tickers , 3 , 4/3 * % , 7 / tickers , 5 , 6 / 5 * % , 10 / tickers , 9 / 8 * %

    # average long only return for ticker, long only bets, long only win ratio, average other long return for ticker, other long bets, other long win ratio
    # 11 / tickers, 12, 13 / 12 * %, 14 / ticker, 15, 16 / 15 * %
    # print(len(obs_mode_data))
    # print(len(rule_data))
    for i in range(len(rule_data)):
        return_list[i].extend(
            [
                (rule_data[i][0]),  # / ticker_length)
                rule_data[i][3],
                (
                    (rule_data[i][4] * 100 / rule_data[i][3])
                    if rule_data[i][3] != 0
                    else 0
                ),
                (rule_data[i][7]),  # / ticker_length)
                rule_data[i][5],
                (
                    (rule_data[i][6] * 100 / rule_data[i][5])
                    if rule_data[i][5] != 0
                    else 0
                ),
                (rule_data[i][10]),  # / ticker_length)
                (
                    (rule_data[i][9] * 100 / rule_data[i][8])
                    if rule_data[i][8] != 0
                    else 0
                ),
                # (rule_data[i][11]), # / ticker_length)
                # rule_data[i][12],
                # (rule_data[i][13] * 100 / rule_data[i][12]) if rule_data[i][12] != 0 else 0,
                # (rule_data[i][14]), # / ticker_length)
                # rule_data[i][15],
                # (rule_data[i][16] * 100 / rule_data[i][15]) if rule_data[i][15] != 0 else 0,
            ]
        )

    return return_list


def get_best_results(sys_crossover_list):
    best_results_index = 0
    for i in range(1, len(sys_crossover_list) - 1):
        # print(sys_crossover_list[i][0], sys_crossover_list[i-1][0])
        if sys_crossover_list[i][0] > sys_crossover_list[best_results_index][0]:
            best_results_index = i
    return sys_crossover_list[best_results_index]


def table_data_generator(data, indexes, len_return_list):
    output_data = []
    for d in data:
        current_return_list = get_data_rule(d, len_return_list)
        output_data.extend(current_return_list)

    # naked_cols = ['net', 'netbet', 'netpercent', 'bets', 'wins', 'longs', 'long_wins', 'long_profit',
    # 'shorts', 'short_wins', 'short_profit', "long_only_net", "long_only_bets", "long_only_wins",
    # "long_only_net_other", "long_only_bets_other", "long_only_wins_other", 'his']

    tempdf = pd.DataFrame(
        output_data,
        columns=[
            "Return",
            "Transactions",
            "Win Ratio",
            "Long Return",
            "Long Transactions",
            "Long Win Ratio",
            "Short Return",
            "Short Win Ratio",
            # "Long 1 Return",
            # "Long 1 Transactions",
            # "Long 1 Win Ratio",
            # "Long 2 Return",
            # "Long 2 Transactions",
            # "Long 2 Win Ratio",
        ],
        index=indexes,
    )

    return tempdf


alpha_range = [
    -0.9,
    -0.8,
    -0.7,
    -0.6,
    -0.5,
    -0.4,
    -0.3,
    -0.2,
    -0.1,
    0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]
gamma_range = [
    1,
    2,
    4,
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    200,
    300,
    400,
    500,
    600,
    700,
    800,
    900,
    1000,
    10000,
    100000,
]
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)

In [2]:
market = "nasdaq"
time_format = "1_day"

macd_params = (12, 26, 9)
# macd_params = (7, 10, 15)

params_str = "_".join(str(val) for val in macd_params)
short_period, long_period, signal_period = macd_params

trade_cost = False
balance = 5036790
risk_per_trade = 100 / 100
last_close = True

prefix = f"./data/{market}/"

ndq_read= pd.read_csv(
    prefix + "ndq.csv",
    index_col="Date",
    parse_dates=True,
    sep=",",
    usecols=["Date", "Open", "High", "Low", "Close"],
)

ndq_read.dropna(inplace=True)
ndq_read.columns = list(map(lambda x: x.lower(), list(ndq_read.columns)))
ndq_read.sort_index(inplace=True)
ndq_read.index.names = ["date"]

ndq_data = calculate_indicators(
    ndq_read,
    col="close",
    ema_mode="normal",
    short_period=short_period,
    long_period=long_period,
    signal_period=signal_period,
    alpha_range=alpha_range,
    gamma_range=gamma_range,
)

ndq_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2009-07-01,1846.12,1861.62,1843.79,1845.72,18.917615,20.614165,-1.696551,21.791840,20.738860,1.052980,...,-4.362060,14306,26.490877,30.852937,-4.362060,17809,26.490877,30.852937,-4.362060,20703
2009-07-02,1823.69,1823.91,1795.95,1796.52,15.375644,19.566461,-4.190817,9.601068,18.511302,-8.910234,...,-4.095509,14306,25.733551,29.829060,-4.095509,17809,25.733551,29.829060,-4.095509,20703
2009-07-06,1783.50,1793.42,1770.11,1787.40,11.697854,17.992739,-6.294886,13.584961,17.526033,-3.941072,...,-4.105040,14306,24.697760,28.802800,-4.105040,17809,24.697760,28.802800,-4.105040,20703
2009-07-07,1787.44,1788.97,1745.38,1746.17,5.394082,15.473008,-10.078926,-1.977709,13.625285,-15.602994,...,-4.324024,14306,23.397770,27.721794,-4.324024,17809,23.397770,27.721794,-4.324024,20703
2009-07-08,1756.00,1757.67,1727.05,1747.17,0.473531,12.473113,-11.999582,2.679647,11.436157,-8.756511,...,-4.899514,14306,21.597401,26.496915,-4.899514,17809,21.597401,26.496915,-4.899514,20703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-17,17722.55,17925.94,17645.88,17808.66,-522.928566,-448.025779,-74.902787,-530.004926,-456.705924,-73.299002,...,-112.052989,14306,-289.233846,-177.180858,-112.052989,17809,-289.233846,-177.180858,-112.052989,20703
2025-03-18,17662.57,17682.92,17431.67,17504.12,-524.731893,-463.367002,-61.364891,-519.986164,-469.361972,-50.624192,...,-108.337969,14306,-312.603318,-204.265350,-108.337969,17809,-312.603318,-204.265350,-108.337969,20703
2025-03-19,17590.12,17917.54,17533.94,17750.79,-500.487527,-470.791107,-29.696420,-482.938754,-472.077329,-10.861426,...,-103.640661,14306,-333.816176,-230.175515,-103.640661,17809,-333.816176,-230.175515,-103.640661,20703
2025-03-20,17586.06,17903.20,17576.82,17691.63,-480.508408,-472.734567,-7.773841,-478.321096,-473.326082,-4.995014,...,-96.246237,14306,-350.483311,-254.237074,-96.246237,17809,-350.483311,-254.237074,-96.246237,20703


In [3]:
two_years = 24

start_optimize = pd.Timestamp("2013-01-01")
end_optimize = pd.Timestamp("2015-06-30")
start_test = pd.Timestamp("2015-07-01")
end_test = pd.Timestamp("2017-12-31")

optimize_data = ndq_data[start_optimize:end_optimize]
test_data = ndq_data[start_test:end_test]

# optimize_data
test_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2015-07-01,5029.05,5038.40,4994.58,5013.12,-4.445838,10.107655,-14.553493,-10.568103,7.918457,-18.486561,...,-1.533026,14306,15.662311,17.195336,-1.533026,17809,15.662311,17.195336,-1.533026,20703
2015-07-02,5024.30,5027.18,4990.77,5009.21,-7.903079,6.505509,-14.408587,-5.504556,5.233855,-10.738411,...,-2.835072,14306,13.651496,16.486568,-2.835072,17809,13.651496,16.486568,-2.835072,20703
2015-07-06,4963.79,5020.61,4961.07,4991.94,-11.899343,2.824538,-14.723881,-17.654651,0.656154,-18.310805,...,-3.992424,14306,11.496038,15.488462,-3.992424,17809,11.496038,15.488462,-3.992424,20703
2015-07-07,4993.76,5001.95,4902.42,4997.46,-14.454373,-0.631244,-13.823129,-11.574122,-1.789902,-9.784220,...,-5.065570,14306,9.156500,14.222070,-5.065570,17809,9.156500,14.222070,-5.065570,20703
2015-07-08,4953.98,4965.33,4901.71,4909.76,-23.287466,-5.162488,-18.124977,-33.829475,-8.197816,-25.631659,...,-5.941326,14306,6.795413,12.736738,-5.941326,17809,6.795413,12.736738,-5.941326,20703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-12-22,6958.02,6962.27,6944.45,6959.96,54.461027,48.559129,5.901898,53.746516,49.108382,4.638134,...,-0.172791,14306,47.299643,47.472434,-0.172791,17809,47.299643,47.472434,-0.172791,20703
2017-12-26,6928.93,6942.14,6915.56,6936.25,51.847653,49.216834,2.630819,50.138676,49.314441,0.824235,...,0.434678,14306,48.015782,47.581104,0.434678,17809,48.015782,47.581104,0.434678,20703
2017-12-27,6941.45,6955.38,6931.34,6939.34,49.455776,49.264622,0.191154,48.841167,49.219786,-0.378619,...,0.654292,14306,48.398969,47.744677,0.654292,17809,48.398969,47.744677,0.654292,20703
2017-12-28,6953.38,6954.81,6936.75,6950.16,47.881334,48.987965,-1.106630,47.017485,48.779326,-1.761841,...,0.607978,14306,48.504650,47.896671,0.607978,17809,48.504650,47.896671,0.607978,20703


SYSTEM 1 - NO EXTRA CONDITION

In [4]:
signal_line_if = False
signal_line_if_target = False

Signalline crossover

In [5]:
sys1_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys1_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys1_major_macd_crossover)):
    sys1_major_macd_crossover[i].extend(sys1_major_macd_crossover[i][-3:])
    print(sys1_major_macd_crossover[i], len(sys1_major_macd_crossover[i]))

best_sys1_major_macd_crossover = get_best_results(sys1_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[-268636.7099999953, -5715.674680850964, -5.333490377800054, 47, 19, 24, 13, 708211.7699999999, 23, 6, -976848.4799999995, 722475.169999999, 24, 13, 722475.169999999, 24, 13] 17
[-781105.2999999989, -18165.239534883694, -15.50799814961511, 43, 16, 22, 12, 355785.65000000206, 21, 4, -1136890.9499999983, 405776.6800000025, 22, 12, 405776.6800000025, 22, 12] 17
[-915485.8299999917, -17273.317547169656, -18.175977755673586, 53, 23, 27, 15, 276624.7600000015, 26, 8, -1192110.589999999, 337836.85000000335, 27, 15, 337836.85000000335, 27, 15] 17
[-915485.8299999917, -17273.317547169656, -18.175977755673586, 53, 23, 27, 15, 276624.7600000015, 26, 8, -1192110.589999999, 337836.85000000335, 27, 15, 337836.85000000335, 27, 15] 17
[238498.7900000047, 4676.44686274519, 4.735134679031778, 51, 22, 26, 16, 998110.089999999, 25, 6, -759611.3000000005, 1015455.4200000018, 26, 16, 1015455.4200000018, 26, 16] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
11,-268636.71,-5715.674681,-5.333490,47,19,24,13,708211.77,23,6,-976848.48,722475.17,24,13,Histogram a: 0.2
14,-781105.30,-18165.239535,-15.507998,43,16,22,12,355785.65,21,4,-1136890.95,405776.68,22,12,Histogram a: 0.5
215,-915485.83,-17273.317547,-18.175978,53,23,27,15,276624.76,26,8,-1192110.59,337836.85,27,15,Histogram a: -0.1 g: 20
222,-915485.83,-17273.317547,-18.175978,53,23,27,15,276624.76,26,8,-1192110.59,337836.85,27,15,Histogram a: -0.1 g: 90
9,238498.79,4676.446863,4.735135,51,22,26,16,998110.09,25,6,-759611.30,1015455.42,26,16,Histogram a: 0


Zeroline crossover

In [6]:
sys1_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys1_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys1_major_zero_crossover)):
    sys1_major_zero_crossover[i].extend(sys1_major_zero_crossover[i][-3:])
    print(sys1_major_zero_crossover[i], len(sys1_major_zero_crossover[i]))

best_sys1_major_zero_crossover = get_best_results(sys1_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[-159797.08000001125, -7989.854000000562, -3.1725976266632374, 20, 6, 10, 4, 478024.24999999907, 10, 2, -637821.3300000004, 527748.6000000006, 10, 4, 527748.6000000006, 10, 4] 17
[-91583.64999999758, -4579.182499999879, -1.8182939928009223, 20, 6, 10, 4, 487962.0499999997, 10, 2, -579545.7000000003, 538778.629999998, 10, 4, 538778.629999998, 10, 4] 17
[210415.46000000183, 16185.804615384755, 4.177570635265751, 13, 6, 6, 4, 797503.7300000008, 7, 2, -587088.2699999986, 875344.4500000011, 6, 4, 875344.4500000011, 6, 4] 17
[210415.46000000183, 16185.804615384755, 4.177570635265751, 13, 6, 6, 4, 797503.7300000008, 7, 2, -587088.2699999986, 875344.4500000011, 6, 4, 875344.4500000011, 6, 4] 17
[-239045.55000000075, -11952.277500000037, -4.745990005539257, 20, 6, 10, 4, 427536.5099999992, 10, 2, -666582.0600000004, 484254.2600000007, 10, 4, 484254.2600000007, 10, 4] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,-159797.08,-7989.854000,-3.172598,20,6,10,4,478024.25,10,2,-637821.33,527748.60,10,4,MACD a: -0.1
3,-91583.65,-4579.182500,-1.818294,20,6,10,4,487962.05,10,2,-579545.70,538778.63,10,4,MACD a: -0.6
427,210415.46,16185.804615,4.177571,13,6,6,4,797503.73,7,2,-587088.27,875344.45,6,4,MACD a: 0.8 g: 1
439,210415.46,16185.804615,4.177571,13,6,6,4,797503.73,7,2,-587088.27,875344.45,6,4,MACD a: 0.8 g: 100
9,-239045.55,-11952.277500,-4.745990,20,6,10,4,427536.51,10,2,-666582.06,484254.26,10,4,MACD a: 0


SYSTEM 2 - SIGNAL EXTRA CONDITION

In [7]:
signal_line_if = True
signal_line_if_target = False

Signalline crossover

In [8]:
sys2_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys2_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys2_major_macd_crossover)):
    sys2_major_macd_crossover[i].extend(sys2_major_macd_crossover[i][-3:])
    print(sys2_major_macd_crossover[i], len(sys2_major_macd_crossover[i]))

best_sys2_major_macd_crossover = get_best_results(sys2_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[-538628.4699999997, -41432.95923076921, -10.693883802977686, 13, 6, 7, 4, 450735.2800000007, 6, 2, -989363.7499999997, 420070.2400000002, 7, 4, 420070.2400000002, 7, 4] 17
[-732040.6000000015, -56310.8153846155, -14.533871771505295, 13, 4, 7, 3, 289801.08999999886, 6, 1, -1021841.6900000006, 270520.3800000008, 7, 3, 270520.3800000008, 7, 3] 17
[481767.75999999605, 43797.069090908735, 9.564976105813347, 11, 5, 6, 3, 348490.14999999845, 5, 2, 133277.60999999888, 333649.1399999978, 6, 3, 333649.1399999978, 6, 3] 17
[481767.75999999605, 43797.069090908735, 9.564976105813347, 11, 5, 6, 3, 348490.14999999845, 5, 2, 133277.60999999888, 333649.1399999978, 6, 3, 333649.1399999978, 6, 3] 17
[-422105.0699999975, -32469.620769230576, -8.380438136193835, 13, 8, 7, 5, 529111.3500000002, 6, 3, -951216.4199999997, 504354.83999999985, 7, 5, 504354.83999999985, 7, 5] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
261,-538628.47,-41432.959231,-10.693884,13,6,7,4,450735.28,6,2,-989363.75,420070.24,7,4,Histogram a: 0.1 g: 4
333,-732040.60,-56310.815385,-14.533872,13,4,7,3,289801.09,6,1,-1021841.69,270520.38,7,3,Histogram a: 0.4 g: 4
403,481767.76,43797.069091,9.564976,11,5,6,3,348490.15,5,2,133277.61,333649.14,6,3,Histogram a: 0.7 g: 1
424,481767.76,43797.069091,9.564976,11,5,6,3,348490.15,5,2,133277.61,333649.14,6,3,Histogram a: 0.7 g: 1000
9,-422105.07,-32469.620769,-8.380438,13,8,7,5,529111.35,6,3,-951216.42,504354.84,7,5,Histogram a: 0


Zeroline crossover

In [9]:
sys2_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys2_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys2_major_zero_crossover)):
    sys2_major_zero_crossover[i].extend(sys2_major_zero_crossover[i][-3:])
    print(sys2_major_zero_crossover[i], len(sys2_major_zero_crossover[i]))

best_sys2_major_zero_crossover = get_best_results(sys2_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[-115354.73000000045, -9612.894166666703, -2.290242992064399, 12, 4, 6, 3, 478148.19999999925, 6, 1, -593502.9300000004, 536588.5700000003, 6, 3, 536588.5700000003, 6, 3] 17
[-53740.48999999836, -4478.37416666653, -1.066959114833026, 12, 4, 6, 3, 483837.24999999953, 6, 1, -537577.7400000005, 542680.2699999996, 6, 3, 542680.2699999996, 6, 3] 17
[-53740.48999999836, -4478.37416666653, -1.066959114833026, 12, 4, 6, 3, 483837.24999999953, 6, 1, -537577.7400000005, 542680.2699999996, 6, 3, 542680.2699999996, 6, 3] 17
[-53740.48999999836, -4478.37416666653, -1.066959114833026, 12, 4, 6, 3, 483837.24999999953, 6, 1, -537577.7400000005, 542680.2699999996, 6, 3, 542680.2699999996, 6, 3] 17
[-194831.70000000298, -16235.975000000248, -3.868171990494005, 12, 4, 6, 3, 428261.02999999933, 6, 1, -623092.7300000002, 492831.6200000001, 6, 3, 492831.6200000001, 6, 3] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,-115354.73,-9612.894167,-2.290243,12,4,6,3,478148.20,6,1,-593502.93,536588.57,6,3,MACD a: -0.1
3,-53740.49,-4478.374167,-1.066959,12,4,6,3,483837.25,6,1,-537577.74,542680.27,6,3,MACD a: -0.6
3,-53740.49,-4478.374167,-1.066959,12,4,6,3,483837.25,6,1,-537577.74,542680.27,6,3,MACD a: -0.6
3,-53740.49,-4478.374167,-1.066959,12,4,6,3,483837.25,6,1,-537577.74,542680.27,6,3,MACD a: -0.6
9,-194831.70,-16235.975000,-3.868172,12,4,6,3,428261.03,6,1,-623092.73,492831.62,6,3,MACD a: 0


SYSTEM 3 - Target SIGNAL EXTRA CONDITION

In [10]:
signal_line_if = False
signal_line_if_target = True

Signalline crossover

In [11]:
sys3_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys3_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys3_major_macd_crossover)):
    sys3_major_macd_crossover[i].extend(sys3_major_macd_crossover[i][-3:])
    print(sys3_major_macd_crossover[i], len(sys3_major_macd_crossover[i]))

best_sys3_major_macd_crossover = get_best_results(sys3_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[-538628.4699999997, -41432.95923076921, -10.693883802977686, 13, 6, 7, 4, 450735.2800000007, 6, 2, -989363.7499999997, 420070.2400000002, 7, 4, 420070.2400000002, 7, 4] 17
[-732040.6000000015, -56310.8153846155, -14.533871771505295, 13, 4, 7, 3, 289801.08999999886, 6, 1, -1021841.6900000006, 270520.3800000008, 7, 3, 270520.3800000008, 7, 3] 17
[-481539.73999999743, -37041.51846153826, -9.560449016139197, 13, 5, 7, 3, 489136.8100000005, 6, 2, -970676.55, 452337.68000000063, 7, 3, 452337.68000000063, 7, 3] 17
[-622130.0199999949, -47856.15538461499, -12.351716470212077, 13, 5, 7, 3, 414901.3900000007, 6, 2, -1037031.4099999997, 382834.01000000164, 7, 3, 382834.01000000164, 7, 3] 17
[-422105.0699999975, -32469.620769230576, -8.380438136193835, 13, 8, 7, 5, 529111.3500000002, 6, 3, -951216.4199999997, 504354.83999999985, 7, 5, 504354.83999999985, 7, 5] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
261,-538628.47,-41432.959231,-10.693884,13,6,7,4,450735.28,6,2,-989363.75,420070.24,7,4,Histogram a: 0.1 g: 4
333,-732040.60,-56310.815385,-14.533872,13,4,7,3,289801.09,6,1,-1021841.69,270520.38,7,3,Histogram a: 0.4 g: 4
236,-481539.74,-37041.518462,-9.560449,13,5,7,3,489136.81,6,2,-970676.55,452337.68,7,3,Histogram a: 0 g: 2
222,-622130.02,-47856.155385,-12.351716,13,5,7,3,414901.39,6,2,-1037031.41,382834.01,7,3,Histogram a: -0.1 g: 90
9,-422105.07,-32469.620769,-8.380438,13,8,7,5,529111.35,6,3,-951216.42,504354.84,7,5,Histogram a: 0


Zerocross crossover

In [12]:
sys3_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys3_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
for i in range(len(sys3_major_zero_crossover)):
    sys3_major_zero_crossover[i].extend(sys3_major_zero_crossover[i][-3:])
    print(sys3_major_zero_crossover[i], len(sys3_major_zero_crossover[i]))

best_sys3_major_zero_crossover = get_best_results(sys3_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[-115354.73000000045, -9612.894166666703, -2.290242992064399, 12, 4, 6, 3, 478148.19999999925, 6, 1, -593502.9300000004, 536588.5700000003, 6, 3, 536588.5700000003, 6, 3] 17
[-53740.48999999836, -4478.37416666653, -1.066959114833026, 12, 4, 6, 3, 483837.24999999953, 6, 1, -537577.7400000005, 542680.2699999996, 6, 3, 542680.2699999996, 6, 3] 17
[-53740.48999999836, -4478.37416666653, -1.066959114833026, 12, 4, 6, 3, 483837.24999999953, 6, 1, -537577.7400000005, 542680.2699999996, 6, 3, 542680.2699999996, 6, 3] 17
[-115354.73000000045, -9612.894166666703, -2.290242992064399, 12, 4, 6, 3, 478148.19999999925, 6, 1, -593502.9300000004, 536588.5700000003, 6, 3, 536588.5700000003, 6, 3] 17
[-194831.70000000298, -16235.975000000248, -3.868171990494005, 12, 4, 6, 3, 428261.02999999933, 6, 1, -623092.7300000002, 492831.6200000001, 6, 3, 492831.6200000001, 6, 3] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,-115354.73,-9612.894167,-2.290243,12,4,6,3,478148.20,6,1,-593502.93,536588.57,6,3,MACD a: -0.1
3,-53740.49,-4478.374167,-1.066959,12,4,6,3,483837.25,6,1,-537577.74,542680.27,6,3,MACD a: -0.6
3,-53740.49,-4478.374167,-1.066959,12,4,6,3,483837.25,6,1,-537577.74,542680.27,6,3,MACD a: -0.6
8,-115354.73,-9612.894167,-2.290243,12,4,6,3,478148.20,6,1,-593502.93,536588.57,6,3,MACD a: -0.1
9,-194831.70,-16235.975000,-3.868172,12,4,6,3,428261.03,6,1,-623092.73,492831.62,6,3,MACD a: 0


Getting Tables Detailed

Signalline crossover

In [13]:
sys_modes = ["System 1", "System 2", "System 3"]
# sys_modes = ["System 1", "System 2"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_major_macd_crossover, sys2_major_macd_crossover, sys3_major_macd_crossover]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nasdaq/nasdaq_macdcross_{params_str}.csv", index=True)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                    
System 1 Rule 1 -268636.71            47      40.43    708211.77   
         Rule 2 -781105.30            43      37.21    355785.65   
         Rule 3 -915485.83            53      43.40    276624.76   
         GA     -915485.83            53      43.40    276624.76   
         MACD    238498.79            51      43.14    998110.09   
System 2 Rule 1 -538628.47            13      46.15    450735.28   
         Rule 2 -732040.60            13      30.77    289801.09   
         Rule 3  481767.76            11      45.45    348490.15   
         GA      481767.76            11      45.45    348490.15   
         MACD   -422105.07            13      61.54    529111.35   
System 3 Rule 1 -538628.47            13      46.15    450735.28   
         Rule 2 -732040.60            13      30.77    289801.09   
         Rule 3 -481539.74            13      38.46    489136.81   
         GA     -622130.02            13      38.46    414901.39   
         MACD   -422105.07            13      61.54    529111.35   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Rule 1                 24           54.17    -976848.48   
         Rule 2                 22           54.55   -1136890.95   
         Rule 3                 27           55.56   -1192110.59   
         GA                     27           55.56   -1192110.59   
         MACD                   26           61.54    -759611.30   
System 2 Rule 1                  7           57.14    -989363.75   
         Rule 2                  7           42.86   -1021841.69   
         Rule 3                  6           50.00     133277.61   
         GA                      6           50.00     133277.61   
         MACD                    7           71.43    -951216.42   
System 3 Rule 1                  7           57.14    -989363.75   
         Rule 2                  7           42.86   -1021841.69   
         Rule 3                  7           42.86    -970676.55   
         GA                      7           42.86   -1037031.41   
         MACD                    7           71.43    -951216.42   

                 Short Win Ratio  
first    second                   
System 1 Rule 1            26.09  
         Rule 2            19.05  
         Rule 3            30.77  
         GA                30.77  
         MACD              24.00  
System 2 Rule 1            33.33  
         Rule 2            16.67  
         Rule 3            40.00  
         GA                40.00  
         MACD              50.00  
System 3 Rule 1            33.33  
         Rule 2            16.67  
         Rule 3            33.33  
         GA                33.33  
         MACD              50.00

In [14]:
# sys_modes = ["System 1", "System 2", "System 3"]
sys_modes = ["System 1", "System 2"]
entries = ["Best", "MACD"]
tuples = []

# data = [[best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]], [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]], [best_sys3_major_macd_crossover, sys3_major_macd_crossover[-1]]]
data = [
    [best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]],
    [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]],
]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 2)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(
    f"./results/nasdaq/nasdaq_macdcross_overall_{params_str}.csv", index=True
)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                    
System 1 Best   -268636.71            47      40.43    708211.77   
         MACD    238498.79            51      43.14    998110.09   
System 2 Best    481767.76            11      45.45    348490.15   
         MACD   -422105.07            13      61.54    529111.35   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Best                   24           54.17    -976848.48   
         MACD                   26           61.54    -759611.30   
System 2 Best                    6           50.00     133277.61   
         MACD                    7           71.43    -951216.42   

                 Short Win Ratio  
first    second                   
System 1 Best              26.09  
         MACD              24.00  
System 2 Best              40.00  
         MACD              50.00

Zeroline crossover

In [15]:
sys_modes = ["System 1", "System 2", "System 3"]
# sys_modes = ["System 1", "System 2"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_major_zero_crossover, sys2_major_zero_crossover, sys3_major_zero_crossover]
# data = [sys1_major_zero_crossover, sys2_major_zero_crossover]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nasdaq/nasdaq_zerocross_{params_str}.csv", index=True)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                    
System 1 Rule 1 -159797.08            20      30.00    478024.25   
         Rule 2  -91583.65            20      30.00    487962.05   
         Rule 3  210415.46            13      46.15    797503.73   
         GA      210415.46            13      46.15    797503.73   
         MACD   -239045.55            20      30.00    427536.51   
System 2 Rule 1 -115354.73            12      33.33    478148.20   
         Rule 2  -53740.49            12      33.33    483837.25   
         Rule 3  -53740.49            12      33.33    483837.25   
         GA      -53740.49            12      33.33    483837.25   
         MACD   -194831.70            12      33.33    428261.03   
System 3 Rule 1 -115354.73            12      33.33    478148.20   
         Rule 2  -53740.49            12      33.33    483837.25   
         Rule 3  -53740.49            12      33.33    483837.25   
         GA     -115354.73            12      33.33    478148.20   
         MACD   -194831.70            12      33.33    428261.03   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Rule 1                 10           40.00    -637821.33   
         Rule 2                 10           40.00    -579545.70   
         Rule 3                  6           66.67    -587088.27   
         GA                      6           66.67    -587088.27   
         MACD                   10           40.00    -666582.06   
System 2 Rule 1                  6           50.00    -593502.93   
         Rule 2                  6           50.00    -537577.74   
         Rule 3                  6           50.00    -537577.74   
         GA                      6           50.00    -537577.74   
         MACD                    6           50.00    -623092.73   
System 3 Rule 1                  6           50.00    -593502.93   
         Rule 2                  6           50.00    -537577.74   
         Rule 3                  6           50.00    -537577.74   
         GA                      6           50.00    -593502.93   
         MACD                    6           50.00    -623092.73   

                 Short Win Ratio  
first    second                   
System 1 Rule 1            20.00  
         Rule 2            20.00  
         Rule 3            28.57  
         GA                28.57  
         MACD              20.00  
System 2 Rule 1            16.67  
         Rule 2            16.67  
         Rule 3            16.67  
         GA                16.67  
         MACD              16.67  
System 3 Rule 1            16.67  
         Rule 2            16.67  
         Rule 3            16.67  
         GA                16.67  
         MACD              16.67

In [16]:
# sys_modes = ["System 1", "System 2", "System 3"]
sys_modes = ["System 1", "System 2"]
entries = ["Best", "MACD"]
tuples = []

# data = [[best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]], [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]], [best_sys3_major_zero_crossover, sys3_major_zero_crossover[-1]]]
data = [
    [best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]],
    [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]],
]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 2)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(
    f"./results/nasdaq/nasdaq_zerocross_overall_{params_str}.csv", index=True
)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                    
System 1 Best    210415.46            13      46.15    797503.73   
         MACD   -239045.55            20      30.00    427536.51   
System 2 Best    -53740.49            12      33.33    483837.25   
         MACD   -194831.70            12      33.33    428261.03   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Best                    6           66.67    -587088.27   
         MACD                   10           40.00    -666582.06   
System 2 Best                    6           50.00    -537577.74   
         MACD                    6           50.00    -623092.73   

                 Short Win Ratio  
first    second                   
System 1 Best              28.57  
         MACD              20.00  
System 2 Best              16.67  
         MACD              16.67